# Notebook 05 — Sensitivity / Metamodel Analysis

Replicates the response surface analysis from §4.4 of Löhndorf & Minner (2009).

**Goals:**
- Generate a space-filling design over the 10 model parameters (Table 1).
- Run LSAPI-2 for each configuration to obtain the discounted reward $Q$.
- Fit the regression metamodels $\hat{Q}_1$ (main effects, eq. 28) and $\hat{Q}_2$ (interactions, eq. 29).
- Interpret coefficient signs and magnitudes (replicating Tables 4–5).

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from stochastic import StochasticParams
from environment import EnvParams
from lsapi import LSAPIParams, run_lsapi, evaluate_lsapi

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Space-filling design

The paper uses a Faure low-discrepancy sequence. Here we use a scrambled Sobol sequence (similar uniformity properties, available in scipy).

In [ ]:
from scipy.stats.qmc import Sobol

# Parameter bounds from Table 1
param_names = [
    'sigma_Y', 'theta_Y', 'sigma_P', 'theta_P', 'rho',
    'C', 'eta', 'gamma', 'u', 'o'
]
lower = np.array([1.0, 0.0, 1.0, 0.0, -0.75, 0.0, 0.5, 0.0, 0.5, 0.0])
upper = np.array([4.0, 0.75, 4.0, 0.75, 0.0,  10.0, 1.0, 0.9, 2.0, 0.5])

N_DESIGN = 100  # use 100 for quick demo; paper uses 2000
sampler = Sobol(d=10, scramble=True, seed=0)
unit_samples = sampler.random(N_DESIGN)
design = lower + unit_samples * (upper - lower)

df_design = pd.DataFrame(design, columns=param_names)
print(f"Design matrix: {df_design.shape}")
df_design.describe().round(3)

## 2. Evaluate LSAPI-2 for each configuration

**Note:** with `N=100, M=50` this is fast (~seconds per run). Increase for closer replication.

In [ ]:
rewards = []
lsapi_cfg = LSAPIParams(N=100, M=50, buffer_size=5000, epsilon=0.01, degree=2, n_action_candidates=30)

for i, row in df_design.iterrows():
    eta = row['eta']
    eta_plus = eta_minus = np.sqrt(eta)  # round-trip: eta = eta+ * eta-
    try:
        stoch = StochasticParams(
            mu_Y=5.0, sigma_Y=row['sigma_Y'], theta_Y=row['theta_Y'],
            mu_P=5.0, sigma_P=row['sigma_P'], theta_P=row['theta_P'],
            rho=row['rho'],
        )
        env = EnvParams(
            C=row['C'], eta_plus=eta_plus, eta_minus=eta_minus,
            u=row['u'], o=row['o'], gamma=row['gamma'],
        )
        rng = np.random.default_rng(i)
        approx = run_lsapi(env, stoch, lsapi_cfg, rng=rng, verbose=False)
        r = evaluate_lsapi(approx, env, stoch, T_eval=2000, seed=i)
        rewards.append(r)
    except Exception as e:
        rewards.append(np.nan)
    if (i + 1) % 20 == 0:
        print(f"  Config {i+1}/{N_DESIGN}  reward={rewards[-1]:.2f}")

df_design['Q'] = rewards
df_valid = df_design.dropna()
print(f"\nValid runs: {len(df_valid)}/{N_DESIGN}")
print(f"Reward stats: mean={df_valid['Q'].mean():.2f}, std={df_valid['Q'].std():.2f}")

## 3. Main effects metamodel $\hat{Q}_1$ (eq. 28)

$\hat{Q}_1 = \beta_0 + \beta_1 Z_1 + \beta_3 Z_3 + \beta_5 Z_5 + \beta_6 Z_6 + \beta_9 Z_9 + \beta_{10} Z_{10}$

In [ ]:
from scipy import stats as sp_stats

Z = df_valid[param_names].values
Q = df_valid['Q'].values

# Z indices (1-based in paper -> 0-based here):
# Z1=sigma_Y(0), Z2=theta_Y(1), Z3=sigma_P(2), Z4=theta_P(3), Z5=rho(4)
# Z6=C(5),       Z7=eta(6),     Z8=gamma(7),   Z9=u(8),       Z10=o(9)

X1_cols = [0, 2, 4, 5, 8, 9]  # sigma_Y, sigma_P, rho, C, u, o
X1_names = ['sigma_Y', 'sigma_P', 'rho', 'C', 'u', 'o']
X1 = np.column_stack([np.ones(len(Z)), Z[:, X1_cols]])

reg1 = LinearRegression(fit_intercept=False)
reg1.fit(X1, Q)
Q1_pred = reg1.predict(X1)
r2_1 = r2_score(Q, Q1_pred)

print(f"Q1 model  R² = {r2_1:.3f}  (paper: 0.530)")
print("\nCoefficients:")
print(f"  {'Intercept':<15} {reg1.coef_[0]:>10.3f}")
for name, coef in zip(X1_names, reg1.coef_[1:]):
    print(f"  {name:<15} {coef:>10.3f}")

## 4. Interaction effects metamodel $\hat{Q}_2$ (eq. 29)

In [ ]:
# Build interaction features: Z1*Z2, Z3*Z4*Z5, and Zi*Z6 for i=1..10
Z1Z2    = Z[:, 0] * Z[:, 1]         # sigma_Y * theta_Y
Z3Z4Z5  = Z[:, 2] * Z[:, 3] * Z[:, 4]  # sigma_P * theta_P * rho
ZiZ6    = Z * Z[:, 5:6]             # each param * C

X2 = np.column_stack([X1, Z1Z2, Z3Z4Z5, ZiZ6])
X2_names = X1_names + ['sigma_Y*theta_Y', 'sigma_P*theta_P*rho'] + \
           [f'{n}*C' for n in param_names]

reg2 = LinearRegression(fit_intercept=False)
reg2.fit(X2, Q)
Q2_pred = reg2.predict(X2)
r2_2 = r2_score(Q, Q2_pred)

print(f"Q2 model  R² = {r2_2:.3f}  (paper: 0.898)")

# Show the interaction coefficients
int_coefs = list(zip(X2_names[len(X1_names):], reg2.coef_[len(X1_names):]))
print("\nInteraction coefficients:")
for name, coef in int_coefs:
    print(f"  {name:<25} {coef:>10.3f}")

## 5. Sensitivity bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Main effects
coef_names_1 = ['Intercept'] + X1_names
coef_vals_1  = reg1.coef_
colors_1 = ['seagreen' if c >= 0 else 'tomato' for c in coef_vals_1]
axes[0].barh(coef_names_1, coef_vals_1, color=colors_1, edgecolor='k', linewidth=0.5)
axes[0].axvline(0, color='k', lw=0.8)
axes[0].set_title(f'Main effects $\\hat{{Q}}_1$  ($R^2={r2_1:.2f}$)')
axes[0].set_xlabel('Coefficient')

# Interaction effects (excluding intercept/main effects for clarity)
int_names = [n for n, _ in int_coefs]
int_vals  = [c for _, c in int_coefs]
colors_2  = ['seagreen' if c >= 0 else 'tomato' for c in int_vals]
axes[1].barh(int_names, int_vals, color=colors_2, edgecolor='k', linewidth=0.5)
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_title(f'Interaction effects $\\hat{{Q}}_2 \\setminus \\hat{{Q}}_1$  ($R^2={r2_2:.2f}$)')
axes[1].set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

## 6. Predicted vs actual reward

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, pred, title, r2 in [
    (axes[0], Q1_pred, '$\\hat{Q}_1$', r2_1),
    (axes[1], Q2_pred, '$\\hat{Q}_2$', r2_2),
]:
    ax.scatter(Q, pred, s=15, alpha=0.6, color='steelblue')
    lo, hi = min(Q.min(), pred.min()), max(Q.max(), pred.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8)
    ax.set_xlabel('Actual reward $Q$')
    ax.set_ylabel(f'Predicted {title}')
    ax.set_title(f'{title}  ($R^2={r2:.3f}$)')

plt.tight_layout()
plt.show()